### Import Packages

In [9]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET

### Get Subcellular Location DataFrame

In [10]:
# Specify the file path
HPA_VERSION = "v20"
file_path = f"../hpa-datasets/{HPA_VERSION}/subcellular_location.tsv"

# Read the TSV file into a DataFrame
genes_df = pd.read_csv(file_path, sep="\t")

In [11]:
genes_df.head()

,Gene,Gene name,Reliability,Main location,Additional location,Extracellular location,Enhanced,Supported,Approved,Uncertain,Single-cell variation intensity,Single-cell variation spatial,Cell cycle dependency,GO id
0,ENSG00000000003,TSPAN6,Approved,Cell Junctions;Cytosol,Nucleoli fibrillar center,NaN,NaN,NaN,Cell Junctions;Cytosol;Nucleoli fibrillar center,NaN,Cytosol,NaN,NaN,Cell Junctions (GO:0030054);Cytosol (GO:000582...
1,ENSG00000000457,SCYL3,Uncertain,Microtubules,Nuclear bodies,NaN,NaN,NaN,NaN,Microtubules;Nuclear bodies,NaN,NaN,NaN,Microtubules (GO:0015630);Nuclear bodies (GO:0...
2,ENSG00000000460,C1orf112,Approved,Mitochondria,NaN,NaN,NaN,NaN,Mitochondria,NaN,NaN,NaN,NaN,Mitochondria (GO:0005739)
3,ENSG00000000938,FGR,Approved,Plasma membrane,Aggresome,NaN,NaN,NaN,Aggresome;Plasma membrane,NaN,NaN,NaN,NaN,Aggresome (GO:0016235);Plasma membrane (GO:000...
4,ENSG00000000971,CFH,Approved,Vesicles,NaN,Predicted to be secreted,NaN,NaN,Vesicles,NaN,NaN,NaN,NaN,Vesicles (GO:0043231)


### Get Target Data from Single Gene

In [12]:
# Fetch the XML
url = f"https://{HPA_VERSION}.proteinatlas.org/ENSG00000000003.xml"
response = requests.get(url)

# Parse the XML
root = ET.fromstring(response.content)

# Find the <cellExpression> element
cell_expr = root.find(".//cellExpression")

# Extract sub-elements
summary = cell_expr.find("summary").text.strip()
verification_type = cell_expr.find("verification").attrib.get("type")
verification = cell_expr.find("verification").text.strip()

image_url = cell_expr.find(".//imageUrl").text

locations = []
for loc in cell_expr.findall(".//location"):
    locations.append({
        "status": loc.attrib.get("status"),
        "GOId": loc.attrib.get("GOId"),
        "location": loc.text
    })

# Display extracted data
print("Summary:", summary)
print("Verification Type:", verification_type)
print("Verification:", verification)
print("Image URL:", image_url)
print("Locations:")
for loc in locations:
    print(f" - {loc['status']} → {loc['location']} (GO: {loc['GOId']})")

Summary: Mainly localized to the cytosol & cell junctions. In addition localized to the nucleoli fibrillar center.
Verification Type: reliability
Verification: approved
Image URL: http://images.proteinatlas.org/4109/1843_B2_17_cr5af971a263864_selected.jpg
Locations:
 - main → cell junctions (GO: GO:0030054)
 - main → cytosol (GO: GO:0005829)
 - additional → nucleoli fibrillar center (GO: GO:0001650)


In [13]:
# Fetch the XML
url = f"https://{HPA_VERSION}.proteinatlas.org/ENSG00000000003.xml"
response = requests.get(url)

# Parse the XML
root = ET.fromstring(response.content)

# Find the cellExpression block
cell_expr = root.findall(".//cellExpression")[1]

# Print cell_expr content
# for child in cell_expr:
#     print(child.tag, child.attrib)

# Get verification type
verification = cell_expr.find(".//verification")
verification_type = verification.attrib.get("type") if verification is not None else None
verification = verification.text.strip() if verification is not None else None

# All <data> entries
results = []
for data in cell_expr.findall(".//data"):
    # print("\nData block:")
    # for child in data:
    #     print(" -", child.tag, child.attrib)
    cell_line_elem = data.find("cellLine")
    cell_line = cell_line_elem.text if cell_line_elem is not None else None
    organ = cell_line_elem.attrib.get("organ") if cell_line_elem is not None else None
    cell_id = cell_line_elem.attrib.get("cellosaurusID") if cell_line_elem is not None else None

    # Locations
    locations = [loc.text for loc in data.findall("location")]

    # Sample images (without channel info)
    image_urls = [
        img.find("imageUrl").text
        for img in data.findall(".//image[@imageType='sampleImage']")
        if img.find("imageUrl") is not None
    ]

    results.append({
        "Cell Line": cell_line,
        "Organ": organ,
        "Cellosaurus ID": cell_id,
        "Locations": locations,
        "Images": image_urls
    })

# Output example
print("Verification Type:", verification_type)
print("Verification:", verification)
for res in results:
    print("\nCell Line:", res["Cell Line"])
    print("Organ:", res["Organ"])
    print("Cellosaurus ID:", res["Cellosaurus ID"])
    print("Locations:", ", ".join(res["Locations"]))
    print("Images:")
    for img in res["Images"]:
        print(" -", img)

Verification Type: validation
Verification: approved

Cell Line: CACO-2
Organ: Gastrointestinal tract
Cellosaurus ID: CVCL_0025
Locations: nucleoli fibrillar center, cytosol
Images:
 - http://images.proteinatlas.org/4109/1832_C1_2_blue_red_green.jpg
 - http://images.proteinatlas.org/4109/1832_C1_4_blue_red_green.jpg

Cell Line: RT4
Organ: Kidney & urinary bladder
Cellosaurus ID: CVCL_0036
Locations: nucleoli fibrillar center, cell junctions, cytosol
Images:
 - http://images.proteinatlas.org/4109/1843_B2_17_cr5af971a263864_blue_red_green.jpg
 - http://images.proteinatlas.org/4109/1843_B2_30_cr5af971a2648d6_blue_red_green.jpg

Cell Line: U-2 OS
Organ: Mesenchymal
Cellosaurus ID: CVCL_0042
Locations: cytosol
Images:
 - http://images.proteinatlas.org/4109/23_H11_1_blue_red_green.jpg
 - http://images.proteinatlas.org/4109/23_H11_2_blue_red_green.jpg


### Get Target Data from List of Genes

In [14]:
def get_hpa_data_from_gene(gene_id, hpa_version):
    # Fetch the XML
    url = f"https://{hpa_version}.proteinatlas.org/{gene_id}.xml"
    response = requests.get(url)

    # Parse the XML
    root = ET.fromstring(response.content)

    # Find the cellExpression block
    cell_expr = root.findall(".//cellExpression")
    cell_expr_0 = cell_expr[0] if cell_expr else None
    cell_expr_1 = cell_expr[1] if len(cell_expr) > 1 else None

    # Extract sub-elements
    gen_source = cell_expr_0.attrib.get("source")
    gen_tech = cell_expr_0.attrib.get("technology")
    gen_summary = cell_expr_0.find("summary").text.strip()
    gen_verification_type = cell_expr_0.find("verification").attrib.get("type")
    gen_verification = cell_expr_0.find("verification").text.strip()

    gen_image_url = cell_expr_0.find(".//imageUrl").text

    gen_locations = []
    for loc in cell_expr_0.findall(".//location"):
        gen_locations.append({
            "status": loc.attrib.get("status"),
            "GOId": loc.attrib.get("GOId"),
            "location": loc.text
        })

    # Get verification type
    source = cell_expr_1.attrib.get("source")
    tech = cell_expr_1.attrib.get("technology")
    verification = cell_expr_1.find(".//verification")
    verification_type = verification.attrib.get("type") if verification is not None else None
    verification = verification.text.strip() if verification is not None else None

    image_url = cell_expr_1.find(".//imageUrl").text

    # All <data> entries
    results = []
    for data in cell_expr_1.findall(".//data"):
        cell_line_elem = data.find("cellLine")
        cell_line = cell_line_elem.text if cell_line_elem is not None else None
        organ = cell_line_elem.attrib.get("organ") if cell_line_elem is not None else None
        cell_id = cell_line_elem.attrib.get("cellosaurusID") if cell_line_elem is not None else None

        # Locations
        locations = [
            loc.text + " (" + loc.attrib.get('GOId', '') + ")" for loc in data.findall("location")]

        # Sample images (without channel info)
        image_urls = [
            img.find("imageUrl").text
            for img in data.findall(".//image[@imageType='sampleImage']")
            if img.find("imageUrl") is not None
        ]

        results.append({
            "Cell Line": cell_line,
            "Organ": organ,
            "Cellosaurus ID": cell_id,
            "Locations": locations,
            "Images": image_urls
        })

    # List of dictionaries for each image
    hpa_data = []
    for res in results:
        for img in res["Images"]:
            hpa_data.append({
                "Gene ID": gene_id,
                "General Summary": gen_summary,
                "General Source": gen_source,
                "General Technology": gen_tech,
                "General Verification Type": gen_verification_type,
                "General Verification": gen_verification,
                "General Image URL": gen_image_url,
                "Source": source,
                "Technology": tech,
                "Verification Type": verification_type,
                "Verification": verification,
                "Image URL": image_url,
                "Cell Line": res["Cell Line"],
                "Organ": res["Organ"],
                "Cellosaurus ID": res["Cellosaurus ID"],
                "Location": res["Locations"],
                "Image": img
            })

    return pd.DataFrame(hpa_data)

In [17]:
from tqdm import tqdm  # better visuals in Jupyter

def get_hpa_data(genes):
    """Fetch HPA data for a list of genes."""
    dfs = []
    for gene in tqdm(genes, desc="Fetching HPA data"):
        try:
            df = get_hpa_data_from_gene(gene_id=gene, hpa_version=HPA_VERSION)
            dfs.append(df)
        except Exception as e:
            print(f"Error fetching data for {gene}: {e}")
    return pd.concat(dfs, ignore_index=True)

In [18]:
df = get_hpa_data(genes=genes_df['Gene'].to_list())

Fetching HPA data:  31%|███▏      | 4012/12813 [1:21:08<46:40:54, 19.09s/it]

Error fetching data for ENSG00000124215: syntax error: line 1, column 49


Fetching HPA data: 100%|██████████| 12813/12813 [4:10:48<00:00,  1.17s/it]  


In [19]:
df.to_csv(HPA_VERSION + "_hpa_subcellular_location_expanded.csv", index=False)